In [1]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry


In [2]:
db = lancedb.connect(uri="music_quiz_test")

In [3]:
model = get_registry().get("sentence-transformers").create(
    name="sentence-transformers/static-similarity-mrl-multilingual-v1"
)

In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/serving/mart.parquet")

df = pd.read_parquet(DATA_PATH)

In [5]:
df["search_text"] = (
    "song_name: " + df["track_name"] +
    ", artist_name: " + df["artist_names"] +
    ", genre: " + df["genres"].apply(", ".join) +
    ", album: " + df["album"]
)

class Quiz(LanceModel):
    spotify_url: str
    track_id: str
    artist_id: str
    track_name: str
    release_year: int
    decade: int
    track_popularity: int
    artist_names: str
    artist_popularity: int
    genres: list[str]
    album: str
    
    search_text: str = model.SourceField()

    vector: Vector(model.ndims()) = model.VectorField()

c:\Users\ricka\Documents\Course\examensarbete\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ricka\Documents\Course\examensarbete\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ricka\.cache\huggingface\hub\models--sentence-transformers--static-similarity-mrl-multilingual-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Develope

In [6]:
quiz_table = db.create_table("quiz", schema=Quiz, data=df)

In [ ]:
table_lance = quiz_table.to_pandas()
print(table_lance.columns)


Index(['spotify_url', 'track_id', 'artist_id', 'track_name', 'release_year',
       'decade', 'track_popularity', 'artist_names', 'artist_popularity',
       'genres', 'album', 'search_text', 'vector'],
      dtype='str')


In [ ]:
print(table_lance["vector"])

0         [0.028767617, 0.05712034, -0.020719128, -0.009...
1         [0.00074080843, 0.04620205, -0.0021910996, 0.0...
2         [0.010764694, 0.02715365, -0.027862538, -0.018...
3         [0.037585236, 0.04974725, -0.046197116, -0.011...
4         [0.00757038, 0.033297945, 0.0044388412, -0.000...
                                ...                        
217563    [-0.007421974, 0.09432165, -0.02864347, 0.0194...
217564    [-0.015894761, 0.07747633, -0.009563143, 0.015...
217565    [-0.0046804817, 0.06375885, -0.025724994, 0.00...
217566    [0.00016414945, 0.050621603, -0.014675594, 0.0...
217567    [0.020685975, 0.07247198, -0.0031723438, -0.01...
Name: vector, Length: 217568, dtype: object


AttributeError: 'NoneType' object has no attribute 'head'

In [12]:
query = "ge mig tre låtar av Miley Cyrus"
results = quiz_table.search(query).limit(3).to_pandas()
print(results)

                                         spotify_url                track_id  \
0  https://open.spotify.com/embed/track/27rdGxbav...  27rdGxbavYJeBphck5MZAF   
1  https://open.spotify.com/embed/track/6zegtH6XX...  6zegtH6XXd2PDPLvy1Y0n2   
2  https://open.spotify.com/embed/track/02dRkCEc8...  02dRkCEc8Q5ch4TTcnLxOn   

                artist_id                                         track_name  \
0  5YGY8feqx7naU7z4HrwZM6    Nothing Breaks Like a Heart (feat. Miley Cyrus)   
1  5YGY8feqx7naU7z4HrwZM6  Don’t Call Me Angel (Charlie’s Angels) (with M...   
2  5YGY8feqx7naU7z4HrwZM6    Nothing Breaks Like a Heart (feat. Miley Cyrus)   

   release_year  decade  track_popularity artist_names  artist_popularity  \
0          2018    2010                76  Miley Cyrus                 88   
1          2019    2010                73  Miley Cyrus                 88   
2          2019    2010                66  Miley Cyrus                 88   

  genres                                         